# 01 - Compare the PDF and the original fax image

Start with the unmodified PDF. If the intake system retained an original TIFF, PNG or JPEG, compare that file too. Keep the analyzer and API version fixed so the input representation is the variable.

This notebook uses `prebuilt-layout` to inspect OCR and layout without custom-field inference. It is not a replacement for subsequently testing the application's extraction analyzer. Do not repair a missing minus sign by guessing from the expected amount.

Run independently from this directory. Follow [setup instructions](../README.md); approved local inputs and generated outputs are ignored by Git.

In [ ]:
import hashlib, json, os
from pathlib import Path

import pandas as pd
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.identity import DefaultAzureCredential
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
PDF = Path('../input/statement.pdf')
SOURCE_IMAGE = None  # Set to Path('../input/original-fax.tiff') if available.
ANALYZER_ID = 'prebuilt-layout'
API_VERSION = '2025-11-01'
KEYWORD = 'property tax'
OUT = Path('../output/01-compare')
OUT.mkdir(parents=True, exist_ok=True)
inputs = {'original_pdf': PDF}
if SOURCE_IMAGE is not None:
    inputs['source_image'] = SOURCE_IMAGE
else:
    print('No intake image configured: only the PDF baseline will be analyzed.')
pd.DataFrame([{'case': label, 'file': str(path), 'bytes': path.stat().st_size} for label, path in inputs.items()])

## Reuse the official SDK sample
The client, binary request and polling flow below are adapted from Microsoft's [sample_analyze_binary.py](https://github.com/Azure/azure-sdk-for-python/blob/53023a2cd9adf255cff520a99985fff84dc58d7d/sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_analyze_binary.py). Changes: Entra-only sign-in, interactive settings, OCR/layout analyzer instead of the RAG analyzer, and local result files. Microsoft copyright and MIT terms: [LICENSE](../LICENSE).

The SDK handles authentication and polling. No custom analyzer is created, and no deployment or shared defaults are changed.

In [ ]:
endpoint = os.getenv('CONTENTUNDERSTANDING_ENDPOINT') or os.environ['APP_CONTENT_UNDERSTANDING_ENDPOINT']
client = ContentUnderstandingClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(process_timeout=60),
    api_version=API_VERSION,
)

def analyze_file(path, label):
    file_bytes = path.read_bytes()
    poller = client.begin_analyze_binary(analyzer_id=ANALYZER_ID, binary_input=file_bytes)
    result = poller.result()
    markdown = '\n\n'.join(content.markdown or '' for content in result.contents)
    (OUT / f'{label}.json').write_text(json.dumps(result.as_dict(), indent=2), encoding='utf-8')
    (OUT / f'{label}.md').write_text(markdown, encoding='utf-8')
    evidence = {'case': label, 'file': path.name, 'sha256': hashlib.sha256(file_bytes).hexdigest(),
                'analyzer': ANALYZER_ID, 'api_version': API_VERSION}
    (OUT / f'{label}.input.json').write_text(json.dumps(evidence, indent=2), encoding='utf-8')
    return markdown

## Analyze the configured inputs - Azure calls
This cell submits each configured file once. It makes fresh, billable requests every time it runs. The originals are read, never re-saved. Results stay under `output/`; do not commit them.

In [ ]:
texts = {}
for label, path in inputs.items():
    print('Analyzing', label, 'with', ANALYZER_ID)
    texts[label] = analyze_file(path, label)

## Inspect the affected value
Read the amount and its sign next to the correct label, and compare them with the source pixels. The compact preview includes nearby lines; complete Markdown, source information and available confidence/grounding remain in the saved response. Identical page coverage matters when comparing a PDF with an intake image.

If only the original PDF fails, retain both representations for the investigation. That narrows the symptom; it does not establish why OCR differed. If redacting or rendering removes the failure, do not describe the changed file as a successful reproduction of the original bug.

In [ ]:
for label, text in texts.items():
    print('\n---', label, '---')
    lines = text.splitlines()
    matches = [i for i, line in enumerate(lines) if KEYWORD.casefold() in line.casefold()]
    if not matches:
        print('Label not found. Inspect the complete saved Markdown; no accuracy conclusion was made.')
    for i in matches[:3]:
        print('\n'.join(lines[max(0, i - 1):i + 3]))